# 🔧 題目 6：不動產房價趨勢

> 📄 詳細需求見 `requirements_spec.md`


## Section 0：環境設定


In [ ]:
import pandas as pd
import sqlite3
import os
import json
print('✅ 套件載入完成')


In [ ]:
OPENAI_API_KEY = ""
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]
print("✅" if OPENAI_API_KEY else "⚠️ fallback")


---
## Section 1：Extract


### Step 1-1：讀取 CSV


In [ ]:
# TODO 🟢:

# 相關程式碼：
# df_raw = pd.read_csv("real_estate.csv")
# print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
# df_raw.head()


### Step 1-2：檢查


In [ ]:
# TODO 🟢:

# 相關程式碼：
# print(df_raw.dtypes)
# print("\n", df_raw.isnull().sum())
# print("\n", df_raw.describe())


### Step 1-3：自由探索


In [ ]:
# TODO 🟢:

# 相關程式碼：
# print(df_raw.iloc[:, 0].value_counts().head(10))
# print(f"\n唯一值: {df_raw.iloc[:, 0].nunique()}")


### Step 1-4：SQLite


In [ ]:
# TODO 🟢:

# 相關程式碼：
# conn = sqlite3.connect("pipeline.db")
# df_raw.to_sql("raw_realestate", conn, if_exists="replace", index=False)
# print(f"✅ raw_realestate: {pd.read_sql('SELECT COUNT(*) as n FROM raw_realestate', conn)['n'][0]} 筆")


---
## Section 2：Transform


### Step 2-1：從 raw 讀出


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


### Step 2-2 ~ 2-4：清洗


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


### 🏁 檢查點


In [ ]:
assert (df["總價元"] > 0).all(), "❌ 總價有非正值"
assert "總價萬" in df.columns, "❌ 缺少總價萬"
assert "面積坪" in df.columns, "❌ 缺少面積坪"
print("✅ 通過")
print(f"   {len(df)} 筆, {len(df.columns)} 欄")


### Step 2-5：寫入 cleaned


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


---
## Section 3：SQL


### Step 3-1：各縣市均價


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


### Step 3-2：建物型態分佈


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


### Step 3-3：視覺化


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


### Step 3-5：存結果


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


---
## Section 4：LLM


In [ ]:
import requests
def llm_analyze(text, api_key=None):
    if api_key: return _llm_api(text, api_key)
    return _llm_fallback(text)
def _llm_api(text, api_key):
    prompt = f"""請分析以下台灣不動產區域，回傳 JSON：
{{"area_character": "蛋黃區/蛋白區/郊區/新興區/其他", "insight": "一句話區域分析"}}\n文字：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}, timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except: return _llm_fallback(text)
def _llm_fallback(text):
    t = text
    if any(w in t for w in ["大安","信義","中正","松山","中山"]): cat = "蛋黃區"
    elif any(w in t for w in ["內湖","南港","士林","北投","文山"]): cat = "蛋白區"
    elif any(w in t for w in ["淡水","三峽","鶯歌","林口","五股"]): cat = "郊區"
    elif any(w in t for w in ["青埔","竹北","新莊","板橋"]): cat = "新興區"
    else: cat = "其他"
    return {"area_character": cat, "insight": text[:30] + "..."}
print("✅ LLM Helper")


### Step 4-1：單筆測試


In [ ]:
test = str(df["鄉鎮市區"].iloc[0])
result = llm_analyze(test, OPENAI_API_KEY if OPENAI_API_KEY else None)
print(f"📝 {test[:60]}\n🤖 {result}")


### Step 4-2：批次


In [ ]:
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None
results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    r = llm_analyze(str(row["鄉鎮市區"]), api_key)
    results.append(r)
    if len(results) % 10 == 0: print(f"  {len(results)}/{BATCH_SIZE}")
print(f"✅ {len(results)} 筆")


### Step 4-3：整理 + 寫入


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


---
## Section 5：驗證


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


---
## Section 6：報告


In [ ]:
# TODO 🟡:
# 💡 參考 solution 或 requirements_spec.md


---
## Section 7：打包


In [ ]:
checks = [("pipeline.db","DB"), ("processed","統計"), ("output/report.md","報告")]
for p,d in checks: print(f"  {'✅' if os.path.exists(p) else '❌'} {d}: {p}")
c = sqlite3.connect("pipeline.db")
for t in ["raw_realestate","cleaned_realestate","analyzed_realestate"]:
    try: print(f"  ✅ {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', c)['n'][0]}")
    except: print(f"  ❌ {t}")
c.close()
